# 3 — Flatness-aware optimisation for domain generalisation

Leave-one-domain-out on PACS again, but now with the protocol done properly:
each source domain is split 80/20 into train and validation, and every step
draws one batch of 64 from **each** of the three source domains and
concatenates them, so a single gradient is computed over 192 images spanning
all three styles. 5,000 iterations, `ResNet18`, lr 1e-4, held-out domain
touched only at the end.

Against that fixed backbone and schedule, five things vary:

| | what changes |
|---|---|
| **SGD** | plain `optim.SGD`, no momentum |
| **SGD + frozen BN** | BatchNorm layers held in eval mode, affine parameters frozen, so the normalisation statistics stay at their ImageNet values instead of being re-estimated from the source mixture |
| **Adam** | adaptive per-parameter step sizes |
| **FAD** | flatness-aware descent — zeroth- *and* first-order flatness in one objective |
| **MIRO** | Adam plus a KL penalty pulling the fine-tuned features back toward the frozen pretrained model's |

## Recorded results

Target accuracy, from the runs stored in the original notebook:

| optimiser | → sketch | → art_painting |
|---|---|---|
| SGD | 54.01% | 58.64% |
| SGD + frozen BN | 50.98% | 62.35% |
| Adam | 60.96% | 78.76% (frozen BN) |
| FAD + frozen BN | 53.75% | 67.24% |
| MIRO (Adam) + frozen BN | — | 74.32% |

Read across, not down: the two target columns disagree about almost
everything. Freezing BatchNorm helps art_painting by 3.7 points and hurts
sketch by 3.0; Adam is the best optimiser on both targets by a wide margin.
Nine runs on two targets with one seed each cannot separate a real effect
from seed noise, which is the first thing this table says.

## Why flatness

The premise is that generalisation to an unseen domain tracks the *flatness*
of the minimum the optimiser lands in — a sharp minimum fits the source
distribution's particular curvature and falls apart when the distribution
moves, while a flat one degrades gently.

- **Zeroth-order flatness** is the largest loss increase anywhere in a
  ρ-ball around the parameters. SAM minimises it, by taking the gradient at
  an adversarially perturbed point.
- **First-order flatness** is the largest gradient *norm* in that ball. GAM
  minimises it, which penalises sharp directions that a zeroth-order
  measurement misses because the loss happens to be low at the sampled
  point.

FAD combines them, `R(θ) = α·R⁰(θ) + (1−α)·R¹(θ)`, and estimates both with
finite differences of gradients rather than Hessian-vector products:

```
g0 = ∇L(θ)                                   plain gradient
g1 = ∇L(θ + ρ·g0/‖g0‖)          h0 = g1 - g0    ≈ ∇R⁰
g2 = ∇L(θ + ρ·h0/‖h0‖)
g3 = ∇L(θ + ρ·g2/‖g2‖)          h1 = g3 - g2    ≈ ∇R¹

θ ← θ - lr·( g0 + β(α·h0 + (1-α)·h1) )
```

Four backward passes per step, no second derivatives. ρ = 0.05, α = 0.5,
β = 1.0 here.

## Why MIRO

A model fine-tuned on three domains drifts away from the general-purpose
features it started with. MIRO keeps it close: the channel-wise mean and
variance of `layer3` and `layer4` are read off both the fine-tuned model and
a frozen copy of the pretrained one, treated as diagonal Gaussians, and the
KL between them is added to the loss with weight `miro_lambda`. The
pretrained model acts as an oracle for what domain-general features look
like.

## 3.1 Setup

Shared by all nine runs: data, model construction, the BatchNorm freeze, the
per-domain evaluation pass, and plotting.

In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torchvision.models as models
from torch.distributions import Normal
import matplotlib.pyplot as plt
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
all_domains = ["art_painting", "cartoon", "photo", "sketch"]
root_path = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
num_classes = 7
val_ratio = 0.2
batch_size = 64
num_iterations = 5000
report_interval = 100
learning_rate = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
criterion = nn.CrossEntropyLoss()

def get_dataset_for_domain(root, domain, transform):
    domain_path = os.path.join(root, domain)
    ds = datasets.ImageFolder(domain_path, transform=transform)
    return ds

def build_loaders(target_domain):
    """Held-out target, and an 80/20 train/val split of each source domain."""
    source_domains = [d for d in all_domains if d != target_domain]
    target_dataset = get_dataset_for_domain(root_path, target_domain, transform)
    test_loader = DataLoader(target_dataset, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0, pin_memory=True)
    source_datasets = {}
    for sd in source_domains:
        ds = get_dataset_for_domain(root_path, sd, transform)
        total_len = len(ds)
        val_len = int(math.floor(val_ratio * total_len))
        train_len = total_len - val_len
        train_subset, val_subset = random_split(ds, [train_len, val_len], generator=torch.Generator().manual_seed(42))
        source_datasets[sd] = {'train': train_subset, 'val': val_subset}
    train_loaders = {}
    val_loaders = {}
    for sd in source_domains:
        train_loaders[sd] = DataLoader(source_datasets[sd]['train'], batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0, pin_memory=True)
        val_loaders[sd] = DataLoader(source_datasets[sd]['val'], batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0, pin_memory=True)
    return source_domains, train_loaders, val_loaders, test_loader

def build_model():
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def freeze_bn(m):
    if isinstance(m, nn.BatchNorm2d):
        m.eval()
        for p in m.parameters():
            p.requires_grad = False

def next_source_batch(source_domains, train_loaders, iters_dict):
    """One batch from every source domain, concatenated: 3 x 64 = 192 images."""
    all_x, all_y = [], []
    for sd in source_domains:
        try:
            x_batch, y_batch = next(iters_dict[sd])
        except StopIteration:
            iters_dict[sd] = iter(train_loaders[sd])
            x_batch, y_batch = next(iters_dict[sd])
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        all_x.append(x_batch)
        all_y.append(y_batch)
    return torch.cat(all_x, dim=0), torch.cat(all_y, dim=0)

def evaluate(model, loader):
    correct, total, total_loss = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total_loss += criterion(out, y).item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

def new_history(source_domains):
    return {'train_loss': [], 'train_acc': [],
            'per_domain_train_loss': {sd: [] for sd in source_domains},
            'per_domain_val_loss': {sd: [] for sd in source_domains},
            'per_domain_train_acc': {sd: [] for sd in source_domains},
            'per_domain_val_acc': {sd: [] for sd in source_domains}}

def report(model, iteration, loss_value, acc, source_domains, train_loaders, val_loaders, history, freeze_batchnorm):
    """Full-dataset train and val pass over every source domain."""
    model.eval()
    print(f"[Iter {iteration}] Train Loss: {loss_value:.4f}, Acc: {acc:.4f}")
    for sd in source_domains:
        tl, ta = evaluate(model, train_loaders[sd])
        history['per_domain_train_loss'][sd].append(tl)
        history['per_domain_train_acc'][sd].append(ta)
        vl, va = evaluate(model, val_loaders[sd])
        history['per_domain_val_loss'][sd].append(vl)
        history['per_domain_val_acc'][sd].append(va)
    model.train()
    if freeze_batchnorm:
        model.apply(freeze_bn)

def plot_history(history, source_domains):
    x_axis = list(range(1, len(history['train_loss']) + 1))
    plt.plot(x_axis, history['train_loss'], label="Train Loss (combined)")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Train Loss - Combined")
    plt.show()
    plt.plot(x_axis, history['train_acc'], label="Train Acc (combined)")
    plt.xlabel("Iteration")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.title("Train Accuracy - Combined")
    plt.show()
    steps = list(range(report_interval, num_iterations + 1, report_interval))
    for sd in source_domains:
        plt.plot(steps, history['per_domain_train_loss'][sd], label=f"{sd} - Train Loss")
        plt.plot(steps, history['per_domain_val_loss'][sd], label=f"{sd} - Val Loss")
        plt.xlabel("Iteration")
        plt.ylabel("Loss")
        plt.title(f"Loss Curves - {sd}")
        plt.legend()
        plt.show()
        plt.plot(steps, history['per_domain_train_acc'][sd], label=f"{sd} - Train Acc")
        plt.plot(steps, history['per_domain_val_acc'][sd], label=f"{sd} - Val Acc")
        plt.xlabel("Iteration")
        plt.ylabel("Accuracy")
        plt.title(f"Accuracy Curves - {sd}")
        plt.legend()
        plt.show()

def final_report(model, test_loader, target_domain):
    model.eval()
    test_loss, test_acc = evaluate(model, test_loader)
    print("=======================")
    print(f"Final Evaluation on Target Domain ({target_domain})")
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")
    print("=======================")
    return test_loss, test_acc

## 3.2 ERM — SGD, Adam, and the BatchNorm freeze

One function covers four of the nine runs. `freeze_batchnorm` holds every
`BatchNorm2d` in eval mode and freezes its affine parameters, so the layer
keeps ImageNet's running statistics instead of re-estimating them on the
source mixture. The idea is that those statistics are themselves
domain-specific: re-estimating them on art+photo+cartoon bakes in a
normalisation that sketch does not share.

The freeze has to be re-applied after every `model.train()` — that call
walks the whole module tree and puts the BatchNorm layers back into training
mode.

In [ ]:
def train_erm(target_domain, optimizer_name="sgd", freeze_batchnorm=False):
    set_seed(42)
    source_domains, train_loaders, val_loaders, test_loader = build_loaders(target_domain)
    model = build_model()
    if freeze_batchnorm:
        model.apply(freeze_bn)
    opt_cls = {"sgd": optim.SGD, "adam": optim.Adam}[optimizer_name]
    optimizer = opt_cls(model.parameters(), lr=learning_rate)
    iters_dict = {sd: iter(train_loaders[sd]) for sd in source_domains}
    history = new_history(source_domains)
    model.train()
    if freeze_batchnorm:
        # model.train() puts every BatchNorm module back into training mode, so
        # the freeze has to be re-applied after it -- here and inside report().
        model.apply(freeze_bn)
    for iteration in range(1, num_iterations + 1):
        x_final, y_final = next_source_batch(source_domains, train_loaders, iters_dict)
        optimizer.zero_grad()
        preds = model(x_final)
        loss = criterion(preds, y_final)
        loss.backward()
        optimizer.step()
        acc = (preds.argmax(1) == y_final).sum().item() / y_final.size(0)
        history['train_loss'].append(loss.item())
        history['train_acc'].append(acc)
        if iteration % report_interval == 0:
            report(model, iteration, loss.item(), acc, source_domains,
                   train_loaders, val_loaders, history, freeze_batchnorm)
    plot_history(history, source_domains)
    return final_report(model, test_loader, target_domain)

## 3.3 FAD

Four gradient evaluations per step. `torch.autograd.grad` is used directly
rather than `.backward()`, and the parameter update is applied by hand, so
there is no optimiser state — this is plain SGD on the modified gradient.

In [ ]:
rho = 0.05
alpha = 0.5
beta = 1.0
xi = 1e-6

def compute_norm(grad_list):
    flat = torch.cat([g.flatten() for g in grad_list])
    return torch.norm(flat)

def add_perturbation(model, perturbation):
    with torch.no_grad():
        for p, pert in zip([p for p in model.parameters() if p.requires_grad], perturbation):
            p.add_(pert)

def subtract_perturbation(model, perturbation):
    with torch.no_grad():
        for p, pert in zip([p for p in model.parameters() if p.requires_grad], perturbation):
            p.sub_(pert)

def train_fad(target_domain, freeze_batchnorm=True):
    set_seed(42)
    source_domains, train_loaders, val_loaders, test_loader = build_loaders(target_domain)
    model = build_model()
    if freeze_batchnorm:
        model.apply(freeze_bn)
    iters_dict = {sd: iter(train_loaders[sd]) for sd in source_domains}
    history = new_history(source_domains)
    model.train()
    if freeze_batchnorm:
        model.apply(freeze_bn)
    for iteration in range(1, num_iterations + 1):
        x_final, y_final = next_source_batch(source_domains, train_loaders, iters_dict)
        preds = model(x_final)
        loss = criterion(preds, y_final)
        g_t0 = torch.autograd.grad(loss, [p for p in model.parameters() if p.requires_grad], create_graph=False)
        g_t0_flat = torch.cat([g.flatten() for g in g_t0])
        norm_g_t0 = torch.norm(g_t0_flat) + xi
        perturbation_1 = [rho * g / norm_g_t0 for g in g_t0]
        add_perturbation(model, perturbation_1)
        outputs_perturbed = model(x_final)
        loss_perturbed = criterion(outputs_perturbed, y_final)
        g_t1 = torch.autograd.grad(loss_perturbed, [p for p in model.parameters() if p.requires_grad], create_graph=False)
        subtract_perturbation(model, perturbation_1)
        h_t0 = [g1 - g0 for g1, g0 in zip(g_t1, g_t0)]
        h_t0_flat = torch.cat([h.flatten() for h in h_t0])
        norm_h_t0 = torch.norm(h_t0_flat) + xi
        perturbation_2 = [rho * h / norm_h_t0 for h in h_t0]
        add_perturbation(model, perturbation_2)
        outputs_perturbed = model(x_final)
        loss_perturbed = criterion(outputs_perturbed, y_final)
        g_t2 = torch.autograd.grad(loss_perturbed, [p for p in model.parameters() if p.requires_grad], create_graph=False)
        subtract_perturbation(model, perturbation_2)
        g_t2_flat = torch.cat([g.flatten() for g in g_t2])
        norm_g_t2 = torch.norm(g_t2_flat) + xi
        perturbation_3 = [rho * g / norm_g_t2 for g in g_t2]
        add_perturbation(model, perturbation_3)
        outputs_perturbed = model(x_final)
        loss_perturbed = criterion(outputs_perturbed, y_final)
        g_t3 = torch.autograd.grad(loss_perturbed, [p for p in model.parameters() if p.requires_grad], create_graph=False)
        subtract_perturbation(model, perturbation_3)
        h_t1 = [g3 - g2 for g3, g2 in zip(g_t3, g_t2)]
        modified_grad = [g0 + beta * (alpha * h0 + (1 - alpha) * h1) for g0, h0, h1 in zip(g_t0, h_t0, h_t1)]
        with torch.no_grad():
            for p, mg in zip([p for p in model.parameters() if p.requires_grad], modified_grad):
                p.data -= learning_rate * mg
        acc = (preds.argmax(1) == y_final).sum().item() / y_final.size(0)
        history['train_loss'].append(loss.item())
        history['train_acc'].append(acc)
        if iteration % report_interval == 0:
            report(model, iteration, loss.item(), acc, source_domains,
                   train_loaders, val_loaders, history, freeze_batchnorm)
    plot_history(history, source_domains)
    return final_report(model, test_loader, target_domain)

## 3.4 MIRO

Forward hooks on `layer3` and `layer4` collect the intermediate feature maps
of both models on the same batch; `compute_miro_loss` reduces each map to a
per-channel mean and variance over the spatial dimensions and returns the
mean KL divergence between the two diagonal Gaussians.

This is the simplified form of MIRO. The published method puts a learnable
projection between the two feature spaces before comparing them, and weights
each layer by an estimate of its reliability; here the raw features are
compared directly, which gives the penalty no freedom to align spaces that
differ by a fixed transform.

In [ ]:
miro_lambda = 0.1

class FeatureExtractor(nn.Module):
    def __init__(self, model):
        super(FeatureExtractor, self).__init__()
        self.model = model
        self.features = []
        self.hooks = []
        def hook_fn(module, input, output):
            self.features.append(output)
        self.hooks.append(model.layer3.register_forward_hook(hook_fn))
        self.hooks.append(model.layer4.register_forward_hook(hook_fn))
    def forward(self, x):
        self.features = []
        _ = self.model(x)
        return self.features
def compute_miro_loss(finetuned_features, pretrained_features):
    total_kl = 0.0
    for ft_feat, pt_feat in zip(finetuned_features, pretrained_features):
        ft_mean = ft_feat.mean(dim=(2, 3))
        ft_var = ft_feat.var(dim=(2, 3)) + 1e-6
        ft_logvar = ft_var.log()
        pt_mean = pt_feat.mean(dim=(2, 3))
        pt_var = pt_feat.var(dim=(2, 3)) + 1e-6
        pt_logvar = pt_var.log()
        ft_dist = Normal(ft_mean, (ft_logvar / 2).exp())
        pt_dist = Normal(pt_mean, (pt_logvar / 2).exp())
        kl = torch.distributions.kl_divergence(ft_dist, pt_dist).mean()
        total_kl += kl
    return total_kl / len(finetuned_features)
def train_miro(target_domain, freeze_batchnorm=True):
    set_seed(42)
    source_domains, train_loaders, val_loaders, test_loader = build_loaders(target_domain)
    model = build_model()
    pretrained_model = models.resnet18(pretrained=True)
    pretrained_model = pretrained_model.to(device)
    pretrained_model.eval()
    for p in pretrained_model.parameters():
        p.requires_grad = False
    if freeze_batchnorm:
        model.apply(freeze_bn)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    finetuned_extractor = FeatureExtractor(model)
    pretrained_extractor = FeatureExtractor(pretrained_model)
    iters_dict = {sd: iter(train_loaders[sd]) for sd in source_domains}
    history = new_history(source_domains)
    model.train()
    if freeze_batchnorm:
        # The original applied freeze_bn *before* model.train(), which undoes it;
        # the freeze only took hold from iteration 100, when report() re-applied
        # it. So the first 100 steps updated the BatchNorm running statistics.
        model.apply(freeze_bn)
    for iteration in range(1, num_iterations + 1):
        x_final, y_final = next_source_batch(source_domains, train_loaders, iters_dict)
        optimizer.zero_grad()
        preds = model(x_final)
        cls_loss = criterion(preds, y_final)
        finetuned_features = finetuned_extractor(x_final)
        with torch.no_grad():
            pretrained_features = pretrained_extractor(x_final)
        miro_loss = compute_miro_loss(finetuned_features, pretrained_features)
        total_loss = cls_loss + miro_lambda * miro_loss
        total_loss.backward()
        optimizer.step()
        acc = (preds.argmax(1) == y_final).sum().item() / y_final.size(0)
        history['train_loss'].append(total_loss.item())
        history['train_acc'].append(acc)
        if iteration % report_interval == 0:
            print(f"[Iter {iteration}] Cls: {cls_loss.item():.4f}, MIRO: {miro_loss.item():.4f}")
            report(model, iteration, total_loss.item(), acc, source_domains,
                   train_loaders, val_loaders, history, freeze_batchnorm)
    plot_history(history, source_domains)
    return final_report(model, test_loader, target_domain)

## 3.5 Running them

Each run is ~40 minutes on a single GPU. Uncomment the call at the bottom.

In [ ]:
RUNS = [
    # (label,                       target,          callable)
    ("SGD",                         "sketch",        lambda: train_erm("sketch", "sgd", False)),
    ("SGD + frozen BN",             "sketch",        lambda: train_erm("sketch", "sgd", True)),
    ("Adam",                        "sketch",        lambda: train_erm("sketch", "adam", False)),
    ("FAD + frozen BN",             "sketch",        lambda: train_fad("sketch")),
    ("SGD",                         "art_painting",  lambda: train_erm("art_painting", "sgd", False)),
    ("SGD + frozen BN",             "art_painting",  lambda: train_erm("art_painting", "sgd", True)),
    ("Adam + frozen BN",            "art_painting",  lambda: train_erm("art_painting", "adam", True)),
    ("FAD + frozen BN",             "art_painting",  lambda: train_fad("art_painting")),
    ("MIRO (Adam) + frozen BN",     "art_painting",  lambda: train_miro("art_painting")),
]

# 5,000 iterations at 192 images per step, plus a full pass over six loaders
# every 100 steps -- roughly 40 minutes per run on one GPU, six hours for all
# nine. Pick one.
RUN = 8
label, target, fn = RUNS[RUN]
print(f"{label} -> {target}")
# fn()

## 3.6 Reading the results

**Adam wins on both targets.** 78.76% on art_painting against 62.35% for
SGD with the same BatchNorm treatment, and 60.96% against 54.01% on sketch.
Its training curves show the classic overfitting signature — source training
accuracy above 95%, validation loss rising while validation accuracy holds —
and the original report treated that as a reason to prefer FAD. It is not:
under the leave-one-domain-out protocol, accuracy on the held-out domain
*is* the generalisation measurement, and Adam has the best one. The rising
validation loss is measured on held-out splits of the **source** domains,
which says something about source overfitting and nothing directly about
target transfer.

**FAD does what it claims relative to SGD**, and only that: 67.24% vs
62.35% on art_painting (+4.9), 53.75% vs 50.98% on sketch (+2.8), at four
times the gradient cost per step. Both gains are consistent in sign, which
is worth something, but a single seed per cell cannot establish they are
real.

**MIRO lands between them** at 74.32%, better than SGD and FAD, below Adam.
Its lowest test loss of the nine (0.8053) alongside a lower accuracy than
Adam's (whose test loss is the *highest*, 2.0100) is the interesting
detail: the KL penalty is producing a well-calibrated model, and Adam a
confident and frequently wrong one that is nonetheless right more often.

The report's own explanation for MIRO's modest gain — ResNet18 rather than
ResNet50, `miro_lambda` possibly too small, unstable feature statistics,
and the missing learnable projection — is the right list, and the missing
projection is the one to fix first.

**What would make this table mean something:** multiple seeds per cell with
the spread reported; all four target domains, not two; and model selection
on the source validation splits rather than reporting whatever the final
iteration produced. The scaffolding for the first two is in place — `RUNS`
above enumerates the grid.